![Universidad Iberoamericana del Ecuador](https://raw.githubusercontent.com/andresrubiop/ml-unibe-estudiantes/main/utils/logo-unibe-color.png)

<div style="background:#004890;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0;border-bottom:4px solid #FCC000"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Machine Learning · Sesión 2 · Notebook del estudiante · no calificable</div><div style="font-size:22px;font-weight:700;margin-top:4px">L2.4 · Escalar: scikit-learn, Spark MLlib, PyTorch y altas prestaciones</div><div style="font-size:12px;opacity:.9;margin-top:4px">Posgrado · Maestría en Big Data y Ciencia de Datos · UNIB.E</div></div>

| | |
|---|---|
| **Objetivo** | Saber **cuándo** y **cómo** llevar un modelo de un portátil a una infraestructura de altas prestaciones: medir el costo, paralelizar, usar **Spark MLlib** y **PyTorch**, y cerrar el ciclo con un modelo guardado y documentado. |
| **Duración** | ≈ 75 min |
| **Teoría** | Manual M2 §10–§11 · Animación A2.12 |
| **Datos icónicos** | **Titanic** en Spark (el mismo flujo que ya conoces, con otra herramienta) |
| **Escala realista** | **MNIST completo**: 60 000 imágenes de entrenamiento y 10 000 de prueba, en Spark y en PyTorch; **Fashion-MNIST** como reto |

**Partes:** 1 · medir antes de escalar: tiempo, memoria y núcleos → 2 · Spark MLlib: del Titanic a MNIST completo → 3 · PyTorch: una red convolucional con `DataLoader` → 4 · cerrar el ciclo: guardar, recargar, documentar y llevarlo a un clúster → reto: la misma red en Fashion-MNIST.

**Requisitos:** Java 17 o 21 para Spark (`java -version`), y los extras del proyecto: `uv sync --extra spark --extra dl`. En Google Colab, la celda de Spark instala `pyspark` si hace falta (PyTorch ya viene instalado).

## 0 · Configuración

In [ ]:
import os
import platform
import time
import urllib.request
import warnings

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")     # Spark en modo local: sin avisos de red
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 2026
UNIBE = {"azul": "#004890", "naranja": "#C2410C", "dorado": "#A87900", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UNIBE.values())), "axes.titlecolor": UNIBE["azul"],
    "axes.titleweight": "bold", "axes.edgecolor": UNIBE["gris"], "axes.grid": True, "grid.color": "#E8EDF2",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
CACHE = os.path.expanduser("~/.cache/ml-unibe")
URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"


def descargar(nombre, url=URL):
    """Descarga un archivo público una sola vez y devuelve su ruta local."""
    os.makedirs(CACHE, exist_ok=True)
    ruta = os.path.join(CACHE, nombre)
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(url + nombre, ruta)
    return ruta


d = np.load(descargar("mnist.npz"))
X_full, y_full = d["x_train"].reshape(-1, 784), d["y_train"]
X_test, y_test = d["x_test"].reshape(-1, 784), d["y_test"]
print(f"Python {platform.python_version()} · scikit-learn {sklearn.__version__} · núcleos disponibles: {os.cpu_count()}")
print(f"MNIST: {X_full.shape} entrenamiento, {X_test.shape} prueba · {X_full.nbytes / 1e6:.0f} MB en uint8, "
      f"{X_full.astype(np.float64).nbytes / 1e6:.0f} MB en float64")

## Parte 1 · Medir antes de escalar

«Big Data» no es un número de filas: es cuando los datos o el cálculo **no caben** en tu herramienta habitual (memoria, tiempo o presupuesto). Antes de cambiar de herramienta, mide. Observa ya un dato: MNIST ocupa 47 MB como enteros de 8 bits, pero 376 MB al convertirlo a `float64`. El **tipo de dato** es la primera optimización.

### ¿Cómo crece el tiempo con n?

**Tu turno.** Mide cuánto tarda en entrenar un clasificador lineal con descenso de gradiente estocástico (`SGDClassifier(loss="log_loss", max_iter=5, tol=None)`, es decir, siempre 5 pasadas por los datos) con 3 750, 7 500, 15 000, 30 000 y 60 000 imágenes.

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier

tamaños = [3_750, 7_500, 15_000, 30_000, 60_000]
X32 = (X_full / 255.0).astype(np.float32)
# TODO: llena la lista segundos con el tiempo de fit para cada tamaño (usa time.perf_counter()).
segundos = []
for n in tamaños:
    ...
pend = np.polyfit(np.log(tamaños), np.log(segundos), 1)[0]
plt.figure(figsize=(5.5, 3.4)); plt.loglog(tamaños, segundos, "o-")
plt.title(f"Tiempo de entrenamiento ∝ n^{pend:.1f}"); plt.xlabel("n (imágenes)"); plt.ylabel("segundos"); plt.show()
print("Segundos:", np.round(segundos, 2))

Una pendiente cercana a 1 significa que el tiempo crece **en proporción** a los datos: el doble de datos, el doble de tiempo. Algoritmos como la SVM con kernel crecen entre n² y n³: con 10 veces más datos tardan entre 100 y 1 000 veces más. **Ese** es el momento de cambiar de algoritmo (por ejemplo, `SGDClassifier`, `MiniBatchKMeans`) o de herramienta.

### Paralelismo en un solo equipo: `n_jobs`

Un bosque aleatorio entrena árboles **independientes**: es fácil repartirlos entre núcleos. Medimos la **aceleración** \(S = T_1 / T_p\).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

nucleos = [1, 2, 4] if os.cpu_count() >= 4 else [1, os.cpu_count()]
tiempos = {}
for p in nucleos:
    t0 = time.perf_counter()
    RandomForestClassifier(n_estimators=100, n_jobs=p, random_state=SEED).fit(X32[:20_000], y_full[:20_000])
    tiempos[p] = time.perf_counter() - t0
for p in nucleos:
    print(f"{p} núcleo(s): {tiempos[p]:5.1f} s · aceleración {tiempos[1] / tiempos[p]:.2f}× (ideal {p}×)")

Aquí la aceleración queda muy cerca de la ideal: los árboles son independientes y los datos caben en la memoria que comparten los núcleos. Casi nunca es así de bueno. La **ley de Amdahl** explica el límite: si una fracción \(f\) del trabajo no se puede paralelizar, con \(p\) núcleos la aceleración máxima es \(S = 1 / (f + (1 - f)/p)\). Aunque solo el 10 % sea secuencial, con infinitos núcleos no pasarías de 10×. Paralelizar tiene costos: copiar datos, coordinar y esperar al más lento.

## Parte 2 · Spark MLlib: del Titanic a MNIST completo

**Apache Spark** reparte los datos en **particiones** y las procesa en paralelo en los núcleos de un equipo o en los nodos de un clúster. Su biblioteca **MLlib** ofrece el mismo flujo que scikit-learn, con otros nombres:

| scikit-learn | Spark MLlib |
|---|---|
| `DataFrame` de pandas, en memoria | `DataFrame` de Spark, distribuido y perezoso (*lazy*) |
| `Pipeline`, `fit`, `predict` | `Pipeline`, `fit`, `transform` |
| columnas de `X` | una sola columna `features` de tipo vector (`VectorAssembler`) |
| `GridSearchCV` | `CrossValidator` con `ParamGridBuilder` |

Aquí Spark corre en **modo local** con todos los núcleos (`local[*]`): el mismo código correría en un clúster cambiando solo el `master`.

In [ ]:
# En Colab puede faltar pyspark: se instala una versión compatible con el Java disponible.
import importlib.util
import re
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    salida = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    java = int(re.search(r'version "(?:1\.)?(\d+)', salida).group(1))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark" if java >= 17 else "pyspark<4"])

from pyspark.sql import SparkSession

spark = (SparkSession.builder.master("local[*]").appName("ml-unibe-L2.4")
         .config("spark.ui.enabled", "false").config("spark.ui.showConsoleProgress", "false")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "· paralelismo:", spark.sparkContext.defaultParallelism)

### El ejemplo icónico: Titanic en Spark

Spark lee el CSV directamente, sin pandas. Las transformaciones no se ejecutan hasta que una **acción** (`count`, `show`, `fit`) lo exige.

In [ ]:
ruta_ti = descargar("titanic.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/")
ti = spark.read.csv(ruta_ti, header=True, inferSchema=True).select("survived", "pclass", "sex", "age", "fare", "embarked")
ti.printSchema()
ti.show(5)

**Tu turno.** Arma el pipeline de Spark: imputar edad y tarifa con la mediana (`Imputer`), convertir `sex` y `embarked` en números (`StringIndexer` con `handleInvalid="keep"`) y luego en indicadores (`OneHotEncoder`), juntar todo en `features` (`VectorAssembler`) y entrenar una `LogisticRegression` con `labelCol="survived"`.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression as SparkLR
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import Imputer, OneHotEncoder, StringIndexer, VectorAssembler

tr_ti, te_ti = ti.randomSplit([0.75, 0.25], seed=SEED)
# TODO: define las etapas y el pipeline; entrénalo con tr_ti.
etapas = [...]
modelo_ti = Pipeline(stages=etapas).fit(tr_ti)
pred_ti = modelo_ti.transform(te_ti)
auc = BinaryClassificationEvaluator(labelCol="survived").evaluate(pred_ti)
acc = MulticlassClassificationEvaluator(labelCol="survived", metricName="accuracy").evaluate(pred_ti)
print(f"Titanic en Spark · prueba: AUC = {auc:.3f} · exactitud = {acc:.3f}")
pred_ti.select("sex", "pclass", "age", "survived", "prediction", "probability").show(5, truncate=40)

### Escala realista: MNIST completo en Spark

Spark lee formatos pensados para grandes volúmenes (Parquet, ORC, CSV particionados, LIBSVM). Escribimos MNIST una vez en formato **LIBSVM** (disperso: solo guarda los píxeles distintos de cero) y lo leemos con Spark. Usamos las **60 000** imágenes, no una muestra.

In [ ]:
from sklearn.datasets import dump_svmlight_file

dir_spark = os.path.join(CACHE, "spark")
os.makedirs(dir_spark, exist_ok=True)
rutas = {"train": os.path.join(dir_spark, "mnist_train.libsvm"), "test": os.path.join(dir_spark, "mnist_test.libsvm")}
for parte, (X, y) in {"train": (X_full, y_full), "test": (X_test, y_test)}.items():
    if not os.path.exists(rutas[parte]) or os.path.getsize(rutas[parte]) == 0:
        dump_svmlight_file(X.astype(np.float32), y, rutas[parte], zero_based=False)   # píxeles enteros 0–255: archivo más pequeño
    print(parte, f"{os.path.getsize(rutas[parte]) / 1e6:.0f} MB")

mn_tr = spark.read.format("libsvm").option("numFeatures", 784).load(rutas["train"]).repartition(8).cache()
mn_te = spark.read.format("libsvm").option("numFeatures", 784).load(rutas["test"]).cache()
print("filas:", mn_tr.count(), "+", mn_te.count(), "· particiones:", mn_tr.rdd.getNumPartitions())

**Tu turno.** Entrena en Spark una regresión logística **multinomial** sobre las 60 000 imágenes (`family="multinomial"`, `maxIter=100`, `regParam=0.001`) y mide la exactitud en las 10 000 de prueba. Luego compara con scikit-learn sobre los mismos datos.

In [ ]:
# TODO: lr_spark, acc_spark y t_spark (segundos de entrenamiento).
t0 = time.perf_counter()
lr_spark = ...
t_spark = time.perf_counter() - t0
acc_spark = ...
t0 = time.perf_counter()
lr_sk = LogisticRegression(max_iter=100).fit(X32, y_full)
t_sk = time.perf_counter() - t0
acc_sk = lr_sk.score((X_test / 255.0).astype(np.float32), y_test)
print(f"Spark MLlib  : exactitud {acc_spark:.4f} · {t_spark:5.1f} s")
print(f"scikit-learn : exactitud {acc_sk:.4f} · {t_sk:5.1f} s")

En **un solo equipo**, Spark no es más rápido que scikit-learn: paga el costo de coordinar particiones y de la máquina virtual de Java. Su ventaja aparece cuando los datos **no caben en la memoria de un equipo** o ya viven en un clúster (un *data lake* con terabytes): ahí scikit-learn ni siquiera puede empezar. La regla práctica: si cabe en tu memoria, usa scikit-learn; si no, Spark (o Dask, o Ray).

El clustering también escala. k-means de Spark usa **k-means||**, una versión paralela de k-means++:

In [ ]:
from pyspark.ml.clustering import KMeans as SparkKMeans
from sklearn.metrics import adjusted_rand_score

t0 = time.perf_counter()
km_spark = SparkKMeans(k=10, seed=SEED, maxIter=30).fit(mn_tr)
grupos = km_spark.transform(mn_tr).select("label", "prediction").toPandas()
print(f"k-means en Spark: {time.perf_counter() - t0:.1f} s · ARI frente a los dígitos = "
      f"{adjusted_rand_score(grupos['label'], grupos['prediction']):.3f}")
fig, axs = plt.subplots(1, 10, figsize=(11, 1.4))
for ax, c in zip(axs, km_spark.clusterCenters()):
    ax.imshow(c.reshape(28, 28), cmap="Greys"); ax.axis("off")
plt.suptitle("Centroides de k-means calculados por Spark sobre 60 000 imágenes", y=1.1); plt.show()

In [ ]:
spark.stop()     # libera la máquina virtual de Java

## Parte 3 · PyTorch: una red convolucional con `DataLoader`

Las redes neuronales escalan de otra forma: se entrenan por **mini-lotes** (nunca necesitan todos los datos a la vez en memoria) y su cálculo, que es álgebra de matrices, vuela en **GPU**. En la tutoría LT.1 viste el bucle de entrenamiento de PyTorch con perceptrones y una red convolucional (CNN); aquí la entrenamos sobre MNIST completo y miramos lo que importa para escalar.

Un `DataLoader` entrega lotes barajados; con `num_workers > 0` y `pin_memory=True` prepara los siguientes lotes **mientras** la GPU entrena, para que no espere. El mismo código corre en CPU o en GPU cambiando solo el `dispositivo`.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "· dispositivo:", dispositivo, "· hilos de CPU:", torch.get_num_threads())
x_tr = torch.tensor(X_full.reshape(-1, 1, 28, 28), dtype=torch.float32) / 255.0
x_te = torch.tensor(X_test.reshape(-1, 1, 28, 28), dtype=torch.float32) / 255.0
t_tr, t_te = torch.tensor(y_full, dtype=torch.long), torch.tensor(y_test, dtype=torch.long)
cargador = DataLoader(TensorDataset(x_tr, t_tr), batch_size=128, shuffle=True,
                      generator=torch.Generator().manual_seed(SEED))


def entrenar(modelo, cargador, epocas=3):
    """El bucle de PyTorch: hacia adelante, pérdida, retropropagación y paso de Adam."""
    modelo.to(dispositivo).train()
    opt = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    for _ in range(epocas):
        for xb, yb in cargador:
            xb, yb = xb.to(dispositivo), yb.to(dispositivo)
            perdida = nn.functional.cross_entropy(modelo(xb), yb)
            opt.zero_grad(); perdida.backward(); opt.step()
    return modelo


def predecir(modelo, X, lote=2048):
    """Clase predicha para cada imagen, sin calcular gradientes."""
    modelo.eval()
    with torch.no_grad():
        return torch.cat([modelo(X[i:i + lote].to(dispositivo)).argmax(1).cpu() for i in range(0, len(X), lote)])

**Tu turno.** Completa `crear_cnn`: dos bloques `Conv2d` + `ReLU` + `MaxPool2d(2)` (16 y 32 filtros de 3 × 3), luego `Flatten`, `Dropout(0.3)` y `Linear(32 * 5 * 5, 10)`. Entrena 3 épocas.

In [ ]:
# TODO: define crear_cnn() con nn.Sequential.
def crear_cnn():
    return nn.Sequential(
        ...
    )


torch.manual_seed(SEED)
cnn = crear_cnn()
t0 = time.perf_counter()
entrenar(cnn, cargador, epocas=3)
t_cnn = time.perf_counter() - t0
acc_cnn = (predecir(cnn, x_te) == t_te).float().mean().item()
n_par = sum(p.numel() for p in cnn.parameters())
print(f"{n_par:,} parámetros · {t_cnn:.0f} s para 3 épocas · exactitud en prueba = {acc_cnn:.4f}")
assert acc_cnn > 0.97

In [ ]:
tabla = pd.DataFrame([
    {"modelo": "logística (scikit-learn)", "herramienta": "scikit-learn", "exactitud": acc_sk, "segundos": t_sk},
    {"modelo": "logística multinomial", "herramienta": "Spark MLlib", "exactitud": acc_spark, "segundos": t_spark},
    {"modelo": "red convolucional", "herramienta": "PyTorch", "exactitud": acc_cnn, "segundos": t_cnn},
])
print(tabla.round(4).to_string(index=False))

La CNN gana con claridad: no por la herramienta, sino porque su arquitectura **entiende la estructura de las imágenes**. La herramienta decide **dónde y a qué escala** puedes entrenar; el modelo decide **qué tan bien** aprendes.

Para varias GPU o varios nodos, PyTorch usa **`DistributedDataParallel`** (DDP): se lanza un proceso por GPU con `torchrun --nproc_per_node=4 entrenar_cnn.py`, cada proceso recibe una parte de cada lote (con `DistributedSampler`) y los gradientes se promedian entre todos antes de cada paso. El bucle de entrenamiento casi no cambia.

## Parte 4 · Cerrar el ciclo: guardar, recargar, documentar y llevarlo a un clúster

Un modelo que solo vive en tu notebook no sirve a nadie (A2.12). Hay que **guardarlo**, poder **recargarlo** en otro lugar y **documentarlo**. En PyTorch se guardan los pesos (`state_dict`) y se recrea la arquitectura con el mismo código.

**Tu turno.** Guarda los pesos de la CNN con `torch.save`, crea una CNN nueva con `crear_cnn()`, carga los pesos con `load_state_dict` y comprueba que predice exactamente lo mismo.

In [ ]:
dir_modelos = os.path.join(CACHE, "modelos")
os.makedirs(dir_modelos, exist_ok=True)
ruta_cnn = os.path.join(dir_modelos, "cnn_mnist.pt")
# TODO: guarda los pesos de cnn en ruta_cnn, recárgalos en cnn2 y compara las salidas en las 100 primeras imágenes de prueba.
torch.save(...)
cnn2 = ...
...
iguales = ...
print(f"Pesos guardados ({os.path.getsize(ruta_cnn) / 1e3:.0f} kB) · predicciones idénticas tras recargar: {iguales}")
assert iguales

Y una **ficha del modelo** (*model card*; Mitchell y colaboradores, 2019): qué hace, con qué datos se entrenó, cómo rinde y dónde **no** debe usarse.

In [ ]:
ficha = f"""# Ficha del modelo · CNN para dígitos manuscritos
- Tarea: clasificar dígitos de 0 a 9 en imágenes de 28 × 28 en escala de grises.
- Datos: MNIST, 60 000 imágenes de entrenamiento y 10 000 de prueba.
- Arquitectura: 2 bloques convolucionales y una capa lineal; {n_par:,} parámetros.
- Desempeño en prueba: exactitud {acc_cnn:.4f}.
- Entorno: Python {platform.python_version()}, PyTorch {torch.__version__}, semilla {SEED}.
- No usar con: letras, dígitos de otros alfabetos, imágenes en color o fotos sin centrar ni recortar.
- Riesgos: MNIST viene de un grupo acotado de escritores; el desempeño con otras caligrafías no está medido.
"""
with open(os.path.join(dir_modelos, "FICHA_cnn_mnist.md"), "w", encoding="utf-8") as f:
    f.write(ficha)
print(ficha)

### Llevarlo a una infraestructura de altas prestaciones (HPC)

En un clúster de HPC no se abre un notebook: se envía un **trabajo** a una cola, con un gestor como **SLURM**. Este es un guion típico (no lo ejecutamos aquí; los nombres de partición y módulos dependen de cada centro de cómputo):

```bash
#!/bin/bash
#SBATCH --job-name=cnn-mnist
#SBATCH --partition=gpu          # cola con GPU (el nombre cambia en cada centro)
#SBATCH --gres=gpu:1             # una GPU
#SBATCH --cpus-per-task=8        # núcleos para el DataLoader
#SBATCH --mem=32G
#SBATCH --time=01:00:00          # tiempo máximo
module load python               # o un contenedor con las versiones exactas
srun python entrenar_cnn.py --epocas 20 --semilla 2026
```

Y para Spark en un clúster, el mismo código de la Parte 2 se envía con `spark-submit --master yarn` (o `k8s://…`), sin tocar el pipeline.

**Preguntas para la discusión (Foro 2):**

1. Tu empresa tiene 5 GB de transacciones en un CSV. ¿scikit-learn o Spark? ¿Y si fueran 5 TB repartidos en un *data lake*?
2. Con 4 núcleos casi nunca se logra 4× más velocidad. Da dos razones, y explica por qué el bosque aleatorio se acercó tanto.
3. ¿Qué debería llevar la ficha de un modelo que decide créditos y que no lleva la de MNIST?

## Reto · Fashion-MNIST con la misma CNN

Entrena la misma arquitectura en **Fashion-MNIST** (mismo formato, prendas en lugar de dígitos) y compara su exactitud con la de MNIST. ¿Qué cambiarías para mejorarla?

In [ ]:
import gzip


def cargar_fashion():
    """Fashion-MNIST desde una copia pública (formato IDX comprimido)."""
    def imagenes(nombre):
        with gzip.open(descargar(nombre)) as f:
            return np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28, 1)

    def etiquetas(nombre):
        with gzip.open(descargar(nombre)) as f:
            return np.frombuffer(f.read(), np.uint8, offset=8)
    return (imagenes("train-images-idx3-ubyte.gz"), etiquetas("train-labels-idx1-ubyte.gz"),
            imagenes("t10k-images-idx3-ubyte.gz"), etiquetas("t10k-labels-idx1-ubyte.gz"))


# TODO (reto): carga Fashion-MNIST, crea una CNN igual con crear_cnn(), entrénala 3 épocas y evalúala.
fx_tr, fy_tr, fx_te, fy_te = cargar_fashion()
...
acc_f = ...
print(f"Misma CNN · MNIST = {acc_cnn:.4f} · Fashion-MNIST = {acc_f:.4f}")

## Desafío opcional con IA agéntica · Del portátil al clúster, con un agente

**Objetivo.** Pedirle a un asistente que convierta el entrenamiento de la CNN en un script reproducible para un clúster y revisar que no invente nombres de colas ni versiones.

**Prompt inicial.** Úsalo en la herramienta que prefieras (Claude Code, Codex, Gemini en Colab, ChatGPT…), con este notebook resuelto como contexto. Pide primero un plan y revisa cada paso antes de aprobarlo.

```text
Tengo el notebook resuelto L2.4 (scikit-learn, Spark MLlib en modo local y una CNN de PyTorch sobre MNIST). Muéstrame primero un plan y espera mi aprobación. Luego escribe un script entrenar_cnn.py con argumentos --epocas, --semilla y --salida que entrene la misma CNN con un DataLoader, guarde los pesos (state_dict) y una ficha del modelo en Markdown, y un guion de SLURM que lo ejecute en una GPU con torchrun. Marca como [COMPLETAR] todo lo que dependa del centro de cómputo (partición, módulos, cuenta).
```

**Cómo verificar el resultado**

- El script fija las semillas y escribe en la ficha las versiones reales de Python y PyTorch leídas en tiempo de ejecución.
- El guion de SLURM no inventa nombres de partición ni de módulos: los deja como [COMPLETAR].
- Ejecutaste el script en local con --epocas 1 y comprobaste que el modelo se recarga y predice igual.

**Declara el uso de IA** (como pide el sílabo): herramienta, prompts relevantes, qué verificaste tú y qué corregiste. El desafío es opcional y no se califica; lo que cuenta es que puedas explicar cada decisión.